# Module 1 Worksheet — Prompting, Sampling, Structured Output, Multimodal

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))      # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Zero-shot vs few-shot (concept_notes.md section 1)
`ask()` is defined in the setup cell now, correctly routed per model.

In [ ]:
zero_shot = ask("Classify sentiment as positive, negative, or neutral. One word only.",
                 "The model latency improved a lot after the upgrade.")
few_shot = ask(
    "Classify sentiment as positive, negative, or neutral. One word only.\n\n"
    "Text: The service was terrible.\nLabel: negative\n\n"
    "Text: It was average.\nLabel: neutral\n",
    "Text: The model latency improved a lot after the upgrade.\nLabel:")
print("Zero-shot:", zero_shot)
print("Few-shot :", few_shot)

## 2. Chain-of-thought

In [ ]:
direct = ask("Answer with only the final number.",
             "5 chunks retrieved, 2 irrelevant. Half the relevant ones mention MCP. How many mention MCP?")
cot = ask("Think step by step, end with 'Answer: <n>' on the last line.",
          "5 chunks retrieved, 2 irrelevant. Half the relevant ones mention MCP. How many mention MCP?")
print("Direct:", direct)
print("\nCoT:", cot)

## 3. Fixing unreliable JSON output (this module's teaser problem, solved live)
Run this a few times. Notice inconsistency — fences sometimes present, sometimes not, depending on model mood/sampling.

In [ ]:
import json

def ask_json_unreliable():
    return ask("Recommend a model for summarizing a 50-page PDF, respond in JSON.",
               "Schema: model, use_case, confidence")

def ask_json_fixed():
    system = ('Respond with ONLY valid JSON, no markdown fences, no extra text. '
              'Schema: {"model": str, "use_case": str, "confidence": float}')
    return ask(system, "Recommend a model for summarizing a 50-page PDF.")

def safe_parse(raw):
    cleaned = raw.strip().strip("`").replace("json\n", "").strip()
    return json.loads(cleaned)

raw1 = ask_json_unreliable()
raw2 = ask_json_fixed()
print("Unreliable prompt raw output:", raw1)
print("Fixed prompt raw output:", raw2)

try:
    print("Parsed (fixed):", safe_parse(raw2))
except json.JSONDecodeError as e:
    print("Still failed to parse:", e, "-> add a few-shot example next.")

## 4. Multimodal call
Requires an actual image — base64-encode any local image file to try this. **Corrected:** uses `ask_vision()`, which fixes two bugs in the original `multimodal_chat()` — no client existed for the VL model at all, and the image was sent as a raw base64 string in a plain-text message rather than a proper `image_url` content block.

In [ ]:
import base64

# image_path = "your_chart.png"
# with open(image_path, "rb") as f:
#     img_b64 = base64.b64encode(f.read()).decode()
#
# response = ask_vision(
#     "Describe what this image shows in 2 sentences.",
#     "What does this chart show?",
#     img_b64,
# )
# print(response)
print("Uncomment above and point at a real image file to try the corrected multimodal call.")

## 5. Model comparison
Same prompt, different models — build the habit of looking before picking. **Important:** under the OLD `multimodal_chat()`, this comparison may have silently queried Qwen3-14B for all three rows due to the routing bug. With `ask()`, each row genuinely hits its own model's endpoint now.

In [ ]:
prompt = "Explain the difference between RAG and fine-tuning in 2 sentences."
for name, model in [("Qwen3-14B", MODEL_QWEN3_14B), ("Qwen3-30B", MODEL_QWEN3_30B), ("Mistral", MODEL_MISTRAL)]:
    print(f"--- {name} ---\n{ask('Be concise.', prompt, model=model)}\n")